# 21. Exploración de mejoras de DES-CPL

Este capítulo prueba, de forma rápida, las mejoras de DES-CPL que quedaron abiertas tras las verificaciones del capítulo 20. **Todo aquí es exploratorio y posterior al protocolo pre-registrado** (`protocolo-v1`): ningún resultado de este capítulo cambia el veredicto del capítulo 17, sea cual sea.

| # | Exploración | Pregunta |
|---|---|---|
| E1 | 5 pliegues y 9 modelos a la vez | V4 y V5 mejoraron a DES-CPL por separado. ¿Cuánto ganan juntas? |
| E2 | Competencia por ordenamiento | El capítulo 15 encontró que las diferencias locales entre modelos son de calibración, no de orden. ¿Mejora DES-CPL si mide la competencia local con el AUC en lugar del Brier? |
| E3 | Potencia | ¿Qué diferencia podría detectar la prueba de Nadeau-Bengio con más repeticiones del bucle externo? |
| E4 | Información | ¿Cuánto mueve la AUC-PR añadir información, frente a cambiar el combinador? |

**Regla (fijada antes de ejecutar).** Una variante se considera una vía real hacia la superioridad solo si supera a XGBoost por al menos 0.006 de AUC-PR media con $p < 0.05$ (Nadeau-Bengio unilateral), el mismo umbral de la verificación V4. Cualquier mejora menor se reporta, pero no se interpreta como superioridad. Además, toda variante se compara con los ensambles estáticos (pesos globales y apilamiento) con los mismos modelos: si ellos ganan lo mismo, la mejora no viene de la selección local.

**Ejecución.** Cada celda costosa guarda su resultado en `results/exploracion/` y, si el archivo ya existe, lo lee en lugar de recalcular. Tiempo estimado la primera vez: E1 unos 15 minutos y E2 unos 10 minutos (los dos conjuntos de modelos); E3 y E4 son inmediatas.

In [1]:
from config import *
from IPython.display import display
from scipy.stats import rankdata
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

from src import configuracion as cfg
from src import experimento_des_cpl as exd
from src.des_cpl import RegionCompetencia, perfil, podar, pesos_softmax, combinar
from src.estadistica import prueba_nadeau_bengio
from src.sota import construir_pipeline_sota

cfg.N_JOBS = 4
FOLDS = list(range(5))
SOTA = ("lightgbm", "catboost")
MODELOS_9 = list(exd.MODELOS) + list(SOTA)
EXP = RESULTADOS / "exploracion"
EXP.mkdir(exist_ok=True)
K3 = RESULTADOS / "des_cpl_60"                          # capítulo 13: 3 pliegues, 7 modelos
K5 = RESULTADOS / "verificaciones" / "oof_k5"           # capítulo 20, V5: 5 pliegues, 7 modelos
AMPLIADO = RESULTADOS / "verificaciones" / "conjunto_ampliado"  # capítulo 20, V4: 3 pliegues, 9 modelos
K5_9 = EXP / "k5_ampliado"                              # E1: 5 pliegues, 9 modelos

roles = roles_variables()
predictores = roles["numericas"] + roles["ordinales"] + roles["binarias"] + roles["categoricas"]
train = leer_tabla("diabetes_train")
X, y, grupos = train[predictores], train[roles["objetivo"]], train[roles["identificador"]]
particiones = exd.particiones_externas(X, y, grupos)
sota = pd.read_csv(RESULTADOS / "benchmarks_sota.csv")
oficial = pd.read_csv(RESULTADOS / "des_cpl_evaluacion_60.csv")
for carpeta in (K3, K5, AMPLIADO):
    assert all((carpeta / f"seleccion_fold{f}.csv").exists() for f in FOLDS), f"Falta {carpeta}"
print(f"{len(X):,} encuentros de entrenamiento; carpetas de los capítulos 13 y 20 disponibles")

79,473 encuentros de entrenamiento; carpetas de los capítulos 13 y 20 disponibles


In [2]:
def leer_fold(carpeta, fold):
    carpeta = Path(carpeta)
    return (pd.read_csv(carpeta / f"seleccion_fold{fold}.csv"),
            pd.read_csv(carpeta / f"evaluacion_fold{fold}.csv"))


def por_fold(tabla, metodo, columna="auc_pr"):
    return tabla[tabla["método"] == metodo].set_index("fold")[columna].reindex(FOLDS)


def comparar(tabla, a, b="xgboost", tabla_b=None):
    # Diferencia por fold (a − b) con la t de Nadeau-Bengio unilateral.
    tabla_b = tabla if tabla_b is None else tabla_b
    d = (por_fold(tabla, a) - por_fold(tabla_b, b)).to_numpy()
    nb = prueba_nadeau_bengio(d)
    return {"comparación": f"{a} − {b}", "AUC-PR": por_fold(tabla, a).mean(), "dif. media": d.mean(),
            "gana en folds": f"{int((d > 0).sum())}/{len(d)}", "t": nb["t"], "p (unilateral)": nb["p"],
            "vía real": bool(d.mean() >= 0.006 and nb["p"] < 0.05)}


def evaluar_carpeta(carpeta, nombre, modelos):
    # Todos los métodos en cada fold (solo la variante final de DES-CPL), con la
    # misma semilla por fold que el capítulo 13. Reanuda fold a fold.
    ruta = EXP / f"{nombre}.csv"
    tabla = pd.read_csv(ruta) if ruta.exists() else pd.DataFrame(columns=["fold"])
    for fold in FOLDS:
        if fold in set(tabla["fold"]):
            continue
        inicio = time.perf_counter()
        seleccion, evaluado = leer_fold(carpeta, fold)
        t = exd.evaluar_fold(seleccion, evaluado, list(modelos), semilla=cfg.SEMILLA + fold,
                             variantes=("DES-CPL",))
        t.insert(0, "fold", fold)
        tabla = pd.concat([tabla, t], ignore_index=True)
        tabla.to_csv(ruta, index=False)
        print(f"[listo] {nombre}, fold {fold}: {time.perf_counter() - inicio:.0f} s")
    return tabla.sort_values("fold", kind="stable").reset_index(drop=True)

## 21.1. E1: cinco pliegues y nueve modelos a la vez

V5 (5 pliegues) y V4 (añadir LightGBM y CatBoost) se midieron por separado. Aquí se combinan: a las predicciones fuera de pliegue con 5 pliegues de V5 se añaden las de LightGBM y CatBoost con los mismos 5 pliegues internos (`StratifiedGroupKFold(5, shuffle=True, random_state=42)`, los de `ejecutar_fold`) y su configuración de `results/benchmarks_sota.csv`. El modelo final de cada fold no depende del número de pliegues, así que se toma de V4 y se comprueba que coincide con `benchmarks_sota.csv`.

In [3]:
K5_9.mkdir(exist_ok=True)
control = []
for fold in FOLDS:
    ruta_sel, ruta_ev = K5_9 / f"seleccion_fold{fold}.csv", K5_9 / f"evaluacion_fold{fold}.csv"
    if ruta_sel.exists() and ruta_ev.exists():
        seleccion, evaluado = leer_fold(K5_9, fold)
    else:
        seleccion, evaluado = leer_fold(K5, fold)
    _, ampliado_ev = leer_fold(AMPLIADO, fold)
    entrenamiento, evaluacion = particiones[fold]
    assert np.array_equal(seleccion["fila"], entrenamiento)
    assert np.array_equal(evaluado["fila"], ampliado_ev["fila"])
    X_tr, y_tr, g_tr = X.iloc[entrenamiento], y.iloc[entrenamiento], grupos.iloc[entrenamiento]
    X_ev = X.iloc[evaluacion]
    pliegues = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=cfg.SEMILLA)
                    .split(X_tr, y_tr, groups=g_tr))
    for nombre in SOTA:
        if f"oof_{nombre}" not in seleccion:
            inicio = time.perf_counter()
            parametros = json.loads(sota.query("modelo == @nombre and fold == @fold")["parametros"].iloc[0])
            base = construir_pipeline_sota(nombre, roles).set_params(**parametros)
            oof, de_pliegue = np.full(len(entrenamiento), np.nan), np.zeros(len(evaluacion))
            for ajuste, validacion in pliegues:
                m = clone(base).fit(X_tr.iloc[ajuste], y_tr.iloc[ajuste])
                oof[validacion] = m.predict_proba(X_tr.iloc[validacion])[:, 1]
                de_pliegue += m.predict_proba(X_ev)[:, 1] / len(pliegues)
            seleccion[f"oof_{nombre}"] = oof
            evaluado[f"final_{nombre}"] = ampliado_ev[f"final_{nombre}"].to_numpy()
            evaluado[f"pliegue_{nombre}"] = de_pliegue
            exd._escribir(evaluado, ruta_ev)
            exd._escribir(seleccion, ruta_sel)
            print(f"[listo] fold {fold}, {nombre}: {time.perf_counter() - inicio:.0f} s")
        referencia = sota.query("modelo == @nombre and fold == @fold")["auc_pr"].iloc[0]
        control.append(abs(average_precision_score(evaluado["y"], evaluado[f"final_{nombre}"]) - referencia))
print(f"Control: máx. |AUC-PR final − benchmarks_sota.csv| = {max(control):.1e}")

[listo] fold 0, lightgbm: 21 s


[listo] fold 0, catboost: 45 s


[listo] fold 1, lightgbm: 21 s


[listo] fold 1, catboost: 38 s


[listo] fold 2, lightgbm: 20 s


[listo] fold 2, catboost: 37 s


[listo] fold 3, lightgbm: 39 s


[listo] fold 3, catboost: 66 s


[listo] fold 4, lightgbm: 26 s


[listo] fold 4, catboost: 73 s
Control: máx. |AUC-PR final − benchmarks_sota.csv| = 1.2e-07


In [4]:
e1 = evaluar_carpeta(K5_9, "e1_k5_9modelos", MODELOS_9)
k5 = pd.read_csv(RESULTADOS / "verificaciones" / "evaluacion_k5.csv")
ampliado = pd.read_csv(RESULTADOS / "verificaciones" / "evaluacion_ampliado.csv")
METODOS = ["xgboost", "catboost", "lightgbm", "Pesos globales (Brier)", "Apilamiento logístico", "DES-CPL"]
tabla_e1 = pd.DataFrame({
    "3 pliegues, 7 modelos (oficial)": [por_fold(oficial, m).mean() for m in METODOS],
    "5 pliegues, 7 modelos (V5)": [por_fold(k5, m).mean() for m in METODOS],
    "3 pliegues, 9 modelos (V4)": [por_fold(ampliado, m).mean() for m in METODOS],
    "5 pliegues, 9 modelos (E1)": [por_fold(e1, m).mean() for m in METODOS]}, index=METODOS)
tabla_e1.loc[["catboost", "lightgbm"], ["3 pliegues, 7 modelos (oficial)", "5 pliegues, 7 modelos (V5)"]] = [
    [sota.query("modelo == @m")["auc_pr"].mean()] * 2 for m in ("catboost", "lightgbm")]
display(tabla_e1.round(4))
display(e1[e1["método"] == "DES-CPL"][["fold", "k", "tau", "lambda", "modelos_combinados",
                                       "auc_pr_seleccion", "auc_pr"]].round(4))
pruebas_e1 = pd.DataFrame([comparar(e1, m) for m in ("DES-CPL", "Pesos globales (Brier)", "Apilamiento logístico")]
                          + [comparar(e1, "DES-CPL", b) for b in ("catboost", "Pesos globales (Brier)")])
display(pruebas_e1.round(4))

[listo] e1_k5_9modelos, fold 0: 82 s


[listo] e1_k5_9modelos, fold 1: 78 s


[listo] e1_k5_9modelos, fold 2: 75 s


[listo] e1_k5_9modelos, fold 3: 82 s


[listo] e1_k5_9modelos, fold 4: 81 s


,"3 pliegues, 7 modelos (oficial)","5 pliegues, 7 modelos (V5)","3 pliegues, 9 modelos (V4)","5 pliegues, 9 modelos (E1)"
xgboost,0.2290,0.2290,0.2290,0.2290
catboost,0.2285,0.2285,0.2285,0.2285
lightgbm,0.2253,0.2253,0.2253,0.2253
Pesos globales (Brier),0.2286,0.2304,0.2296,0.2310
Apilamiento logístico,0.2280,0.2291,0.2320,0.2333
DES-CPL,0.2279,0.2289,0.2302,0.2314


,fold,k,tau,lambda,modelos_combinados,auc_pr_seleccion,auc_pr
12,0,5.0,0.0100,0.50,"random_forest, xgboost, lightgbm, catboost",0.2203,0.2534
25,1,30.0,0.0300,0.00,"random_forest, xgboost, lightgbm, catboost",0.2317,0.2275
38,2,50.0,0.0001,1.00,"random_forest, xgboost, lightgbm, catboost",0.2279,0.2203
51,3,20.0,0.0300,0.25,"random_forest, xgboost, lightgbm, catboost",0.2338,0.2245
64,4,20.0,0.0001,0.90,"logistica, bayes, knn, arbol, random_forest, x...",0.2328,0.2314


,comparación,AUC-PR,dif. media,gana en folds,t,p (unilateral),vía real
0,DES-CPL − xgboost,0.2314,0.0024,3/5,0.7727,0.2414,False
1,Pesos globales (Brier) − xgboost,0.2310,0.0021,2/5,0.9832,0.1906,False
2,Apilamiento logístico − xgboost,0.2333,0.0043,4/5,1.5268,0.1008,False
3,DES-CPL − catboost,0.2314,0.0029,3/5,0.7649,0.2435,False
4,DES-CPL − Pesos globales (Brier),0.2314,0.0004,2/5,0.1942,0.4277,False


## 21.2. E2: competencia por ordenamiento local

DES-CPL mide la competencia local con la puntuación de Brier, que premia la calidad de la probabilidad. Pero el capítulo 15 mostró que las diferencias locales de Brier entre modelos son sobre todo de calibración dentro de cada zona, y la métrica objetivo (AUC-PR) solo mide el orden. Aquí se cambia solo la medida de competencia:

$$\hat C_m^{\text{orden}}(x) = 1 - \widehat{\mathrm{AUC}}_m\big(R_k(x)\big), \qquad \widehat{\mathrm{AUC}}_m(R) = \frac{1}{k^2}\sum_{i \in R^+}\sum_{j \in R^-}\Big[\mathbb{1}\{f_m(x_i) > f_m(x_j)\} + \tfrac{1}{2}\mathbb{1}\{f_m(x_i) = f_m(x_j)\}\Big],$$

donde $R_k(x)$ es la misma región estratificada de DES-CPL ($k$ positivos y $k$ negativos sin el propio paciente), así que el AUC local siempre está definido. El resto no cambia: encogimiento hacia el AUC global, pesos $w_m \propto \exp(-\hat C_m/\tau)$, poda a 4 o 7 modelos y 60 evaluaciones de Optuna por AUC-PR en el conjunto de selección. Las diferencias de AUC entre modelos son mucho mayores que las de Brier, así que la rejilla de $\tau$ se desplaza hacia arriba. Con $\lambda = 1$ la variante es un ensamble de pesos globales por AUC.

Se evalúa con el conjunto oficial (3 pliegues, 7 modelos) y con el de E1 (5 pliegues, 9 modelos).

In [5]:
TAU_ORDEN = (0.0, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0, np.inf)


def auc_local(P, vec_neg, vec_pos, k, lote=4000):
    # AUC de cada modelo en la región de cada consulta, (n, M), por rangos
    # (Mann-Whitney): k negativos y k positivos, empates a la mitad.
    n, M = len(vec_neg), P.shape[1]
    salida = np.empty((n, M))
    for a in range(0, n, lote):
        neg, pos = P[vec_neg[a:a + lote, :k]], P[vec_pos[a:a + lote, :k]]
        rangos = rankdata(np.concatenate([neg, pos], axis=1), axis=1)[:, k:, :]
        salida[a:a + lote] = (rangos.sum(axis=1) - k * (k + 1) / 2) / (k * k)
    return salida


def evaluar_orden(seleccion, evaluado, modelos, semilla, ks=exd.K_REJILLA,
                  lambdas=exd.LAMBDA_REJILLA, podas=exd.PODA_REJILLA, taus=TAU_ORDEN):
    # DES-CPL con competencia 1 − AUC local; mismo esquema y búsqueda que la oficial.
    P_sel = seleccion[[f"oof_{m}" for m in modelos]].to_numpy()
    y_sel = seleccion["y"].to_numpy().astype(int)
    g_sel = seleccion["grupo"].to_numpy()
    P_ev = evaluado[[f"final_{m}" for m in modelos]].to_numpy()
    f_ev = perfil(evaluado[[f"pliegue_{m}" for m in modelos]].to_numpy())
    y_ev = evaluado["y"].to_numpy().astype(int)
    k_max, espacios, cache = max(ks), {}, {}

    def espacio(poda):
        if poda not in espacios:
            activos = podar(P_sel, y_sel, poda)
            Ps, fs = P_sel[:, activos], perfil(P_sel[:, activos])
            region = RegionCompetencia(fs, y_sel, g_sel)
            espacios[poda] = {
                "activos": activos, "Ps": Ps, "Pe": P_ev[:, activos],
                "glob": 1 - np.array([[roc_auc_score(y_sel, Ps[:, m]) for m in range(Ps.shape[1])]]),
                "vec_sel": region.vecindario(fs, k_max, g_sel),
                "vec_ev": region.vecindario(f_ev[:, activos], k_max)}
        return espacios[poda]

    def puntuar(poda, k, lam, tau):
        e = espacio(poda)
        if (poda, k) not in cache:
            cache[(poda, k)] = 1 - auc_local(e["Ps"], e["vec_sel"][0], e["vec_sel"][1], k)
        c = (1 - lam) * cache[(poda, k)] + lam * e["glob"]
        return average_precision_score(y_sel, combinar(pesos_softmax(c, tau), e["Ps"]))

    puntaje, config = exd._buscar(puntuar, {"poda": list(podas), "k": [int(k) for k in ks],
                                            "lam": [float(v) for v in lambdas],
                                            "tau": [float(t) for t in taus]},
                                  exd.PRESUPUESTO, semilla)
    e = espacio(config["poda"])
    local_ev = 1 - auc_local(e["Ps"], e["vec_ev"][0], e["vec_ev"][1], config["k"])
    c_ev = (1 - config["lam"]) * local_ev + config["lam"] * e["glob"]
    p = combinar(pesos_softmax(c_ev, config["tau"]), e["Pe"])
    return {"método": "DES-CPL por orden", "familia": "Exploración", "k": config["k"],
            "tau": config["tau"], "lambda": config["lam"],
            "modelos_combinados": ", ".join(modelos[i] for i in e["activos"]),
            "auc_pr_seleccion": puntaje, "auc_pr": average_precision_score(y_ev, p),
            "auc_roc": roc_auc_score(y_ev, p), "brier": float(np.mean((p - y_ev) ** 2))}


def correr_orden(carpeta, nombre, modelos):
    ruta = EXP / f"{nombre}.csv"
    tabla = pd.read_csv(ruta) if ruta.exists() else pd.DataFrame(columns=["fold"])
    for fold in FOLDS:
        if fold in set(tabla["fold"]):
            continue
        inicio = time.perf_counter()
        seleccion, evaluado = leer_fold(carpeta, fold)
        fila = evaluar_orden(seleccion, evaluado, list(modelos), cfg.SEMILLA + fold)
        tabla = pd.concat([tabla, pd.DataFrame([{"fold": fold, **fila}])], ignore_index=True)
        tabla.to_csv(ruta, index=False)
        print(f"[listo] {nombre}, fold {fold}: {time.perf_counter() - inicio:.0f} s")
    return tabla.sort_values("fold", kind="stable").reset_index(drop=True)

In [6]:
orden_k3 = correr_orden(K3, "e2_orden_k3_7modelos", exd.MODELOS)
orden_e1 = correr_orden(K5_9, "e2_orden_k5_9modelos", MODELOS_9)
con_orden_k3 = pd.concat([oficial, orden_k3], ignore_index=True)
con_orden_e1 = pd.concat([e1, orden_e1], ignore_index=True)
for titulo, tabla, datos in (("3 pliegues, 7 modelos", orden_k3, con_orden_k3),
                             ("5 pliegues, 9 modelos", orden_e1, con_orden_e1)):
    print(f"\n{titulo}")
    display(tabla[["fold", "k", "tau", "lambda", "modelos_combinados", "auc_pr_seleccion", "auc_pr"]].round(4))
    display(pd.DataFrame([comparar(datos, "DES-CPL por orden"),
                          comparar(datos, "DES-CPL por orden", "DES-CPL"),
                          comparar(datos, "DES-CPL por orden", "Pesos globales (Brier)")]).round(4))

[listo] e2_orden_k3_7modelos, fold 0: 56 s


[listo] e2_orden_k3_7modelos, fold 1: 57 s


[listo] e2_orden_k3_7modelos, fold 2: 57 s


[listo] e2_orden_k3_7modelos, fold 3: 62 s


[listo] e2_orden_k3_7modelos, fold 4: 64 s


[listo] e2_orden_k5_9modelos, fold 0: 67 s


... (5 outputs left out)

,comparación,AUC-PR,dif. media,gana en folds,t,p (unilateral),vía real
0,DES-CPL por orden − xgboost,0.2269,-0.0021,1/5,-0.5437,0.6922,False
1,DES-CPL por orden − DES-CPL,0.2269,-0.0011,2/5,-0.8499,0.7784,False
2,DES-CPL por orden − Pesos globales (Brier),0.2269,-0.0017,2/5,-1.0320,0.8198,False



5 pliegues, 9 modelos


,fold,k,tau,lambda,modelos_combinados,auc_pr_seleccion,auc_pr
0,0,5.0,1.000,0.0,"random_forest, xgboost, lightgbm, catboost",0.2202,0.2542
1,1,50.0,0.000,1.0,"random_forest, xgboost, lightgbm, catboost",0.2291,0.2238
2,2,50.0,0.000,1.0,"random_forest, xgboost, lightgbm, catboost",0.2279,0.2203
3,3,5.0,0.001,1.0,"logistica, bayes, knn, arbol, random_forest, x...",0.2331,0.2215
4,4,30.0,0.000,1.0,"logistica, bayes, knn, arbol, random_forest, x...",0.2334,0.2348


,comparación,AUC-PR,dif. media,gana en folds,t,p (unilateral),vía real
0,DES-CPL por orden − xgboost,0.2309,0.0020,2/5,0.8933,0.2111,False
1,DES-CPL por orden − DES-CPL,0.2309,-0.0005,3/5,-0.2480,0.5918,False
2,DES-CPL por orden − Pesos globales (Brier),0.2309,-0.0001,2/5,-0.7134,0.7425,False


## 21.3. E3: potencia con más repeticiones del bucle externo

Con $r$ repeticiones de la validación cruzada de 5 pliegues, la $t$ corregida usa la varianza de las $5r$ diferencias multiplicada por $\frac{1}{5r} + \frac{n_2}{n_1}$, con $n_2/n_1 = 1/4$ (Nadeau y Bengio, 2003; Bouckaert y Frank, 2004). El segundo término no baja con las repeticiones, así que la potencia tiene un piso. Se calcula la diferencia mínima detectable (α = 0.05 unilateral) con la desviación observada de las diferencias DES-CPL − XGBoost, suponiendo que no cambia al repetir.

In [7]:
d = (por_fold(oficial, "DES-CPL") - por_fold(oficial, "xgboost")).to_numpy()
s = d.std(ddof=1)
filas = []
for r in (1, 2, 5, 10, 50):
    n = 5 * r
    mde = stats.t.ppf(0.95, n - 1) * np.sqrt((1 / n + 0.25) * s ** 2)
    filas.append({"repeticiones": r, "diferencias": n, "diferencia mínima detectable": mde,
                  "cómputo aproximado (h)": 2.0 * r})
filas.append({"repeticiones": "∞", "diferencias": "∞",
              "diferencia mínima detectable": stats.norm.ppf(0.95) * np.sqrt(0.25) * s,
              "cómputo aproximado (h)": np.nan})
potencia = pd.DataFrame(filas)
display(potencia.round(4))
print(f"Desviación de las diferencias por fold: {s:.4f}. Diferencia observada: {d.mean():+.4f}.")

,repeticiones,diferencias,diferencia mínima detectable,cómputo aproximado (h)
0,1,5,0.0084,2.0
1,2,10,0.0064,4.0
2,5,25,0.0054,10.0
3,10,50,0.0051,20.0
4,50,250,0.0049,100.0
5,∞,∞,0.0048,NaN


Desviación de las diferencias por fold: 0.0059. Diferencia observada: -0.0010.


El cómputo aproximado supone unas 2 horas por repetición (búsquedas de 60 evaluaciones de los modelos base, conjunto de selección y búsqueda de DES-CPL en los 5 folds) y es solo orientativo.

## 21.4. E4: información frente a combinador

El capítulo 13 (sección 13.6) midió, con otro modelo y una partición propia, cuánto mejora la AUC-PR al recuperar información que el ETL descartó. Aquí se pone al lado de las mejoras del combinador.

In [8]:
info = pd.read_csv(RESULTADOS / "limite_informativo_resumen.csv")
display(info.round(4))
combinador = pd.DataFrame([
    {"cambio": "5 pliegues (V5)", "DES-CPL − XGBoost": comparar(k5, "DES-CPL")["dif. media"]},
    {"cambio": "9 modelos (V4)", "DES-CPL − XGBoost": comparar(ampliado, "DES-CPL")["dif. media"]},
    {"cambio": "5 pliegues y 9 modelos (E1)", "DES-CPL − XGBoost": comparar(e1, "DES-CPL")["dif. media"]},
    {"cambio": "competencia por orden (E2, E1)", "DES-CPL − XGBoost": comparar(con_orden_e1, "DES-CPL por orden")["dif. media"]},
])
display(combinador.round(4))

,variante,AUC-PR media,dif. vs referencia,pliegues a favor,p (Nadeau-Bengio)
0,+ ICD-9 jerárquico,0.2331,0.0029,4,0.1056
1,+ ICD-9 jerárquico + historial,0.2358,0.0055,5,0.0104
2,+ historial del paciente,0.2322,0.0019,3,0.3134
3,Etiqueta ordinal: puntaje compartido + corrección,0.2285,-0.0017,2,0.7421
4,Etiqueta ordinal: solo «cualquier reingreso»,0.1957,-0.0345,0,0.9975
5,Etiqueta ordinal: tres clases,0.2282,-0.0020,2,0.7617
6,Referencia (variables del ETL),0.2303,0.0000,0,NaN


,cambio,DES-CPL − XGBoost
0,5 pliegues (V5),-0.0001
1,9 modelos (V4),0.0012
2,5 pliegues y 9 modelos (E1),0.0024
3,"competencia por orden (E2, E1)",0.0020


## 21.5. Síntesis

In [9]:
sintesis = pd.DataFrame([
    {"exploración": "E1: 5 pliegues y 9 modelos", **comparar(e1, "DES-CPL")},
    {"exploración": "E1: pesos globales con lo mismo", **comparar(e1, "Pesos globales (Brier)")},
    {"exploración": "E1: apilamiento con lo mismo", **comparar(e1, "Apilamiento logístico")},
    {"exploración": "E2: orden, 3 pliegues y 7 modelos", **comparar(con_orden_k3, "DES-CPL por orden")},
    {"exploración": "E2: orden, 5 pliegues y 9 modelos", **comparar(con_orden_e1, "DES-CPL por orden")},
]).drop(columns="comparación")
sintesis.to_csv(EXP / "sintesis.csv", index=False)
display(sintesis.round(4))
print("Alguna vía real hacia la superioridad:", "sí" if sintesis["vía real"].any() else "no")

,exploración,AUC-PR,dif. media,gana en folds,t,p (unilateral),vía real
0,E1: 5 pliegues y 9 modelos,0.2314,0.0024,3/5,0.7727,0.2414,False
1,E1: pesos globales con lo mismo,0.2310,0.0021,2/5,0.9832,0.1906,False
2,E1: apilamiento con lo mismo,0.2333,0.0043,4/5,1.5268,0.1008,False
3,"E2: orden, 3 pliegues y 7 modelos",0.2269,-0.0021,1/5,-0.5437,0.6922,False
4,"E2: orden, 5 pliegues y 9 modelos",0.2309,0.0020,2/5,0.8933,0.2111,False


Alguna vía real hacia la superioridad: no


### 21.5.1. Lectura

**Ninguna exploración abre una vía hacia la superioridad.** Ninguna variante supera a XGBoost por 0.006 con $p < 0.05$. Con todas las mejoras juntas, DES-CPL llega a 0.2314, 0.0024 sobre XGBoost ($p = 0.24$), y aun así queda detrás del apilamiento con los mismos modelos.

**E1. Cinco pliegues y nueve modelos.**
- Las dos mejoras se suman: DES-CPL pasa de 0.2279 (oficial) a 0.2289 con 5 pliegues, 0.2302 con 9 modelos y 0.2314 con ambas cosas. Gana a XGBoost en 3 de 5 folds ($p = 0.24$).
- Los ensambles estáticos ganan lo mismo o más: los pesos globales llegan a 0.2310 y el apilamiento a 0.2333 ($+0.0043$, 4 de 5 folds, $p = 0.10$). Frente a los pesos globales, DES-CPL gana solo 0.0004 ($p = 0.43$).
- Un matiz nuevo: con modelos más fuertes y diversos, la búsqueda usa más la parte local ($\lambda$ = 0.5, 0, 1, 0.25 y 0.9, frente a valores casi siempre de 0.9 o 1 en la versión oficial). Aun así, eso no se traduce en una ventaja sobre la combinación global. La mejora viene de combinar mejores modelos, no de la selección local.

**E2. Competencia por ordenamiento.** La hipótesis no se confirma.
- Con la configuración oficial, la competencia por AUC local queda en 0.2269, por debajo de la de Brier (−0.0011) y de XGBoost (−0.0021).
- Con la de E1, llega a 0.2309, prácticamente igual que la de Brier (−0.0005) y que los pesos globales (−0.0001).
- La búsqueda elige $\lambda = 1$ en 7 de los 10 folds: tampoco con una medida de orden aparece heterogeneidad local aprovechable. Es la misma conclusión del diagnóstico del capítulo 13 por otra vía: no es la medida de competencia la que falla, sino que no hay señal sobre qué modelo ordena mejor a cada paciente.

**E3. Potencia.**
- Con la variabilidad observada, la diferencia mínima detectable es 0.0084 con una repetición, 0.0064 con 2, 0.0054 con 5 y 0.0051 con 10. El piso, con infinitas repeticiones, es 0.0048, porque el término $n_2/n_1 = 1/4$ de la corrección no baja al repetir.
- Ninguna de las mejoras medidas aquí (la mayor, la del apilamiento, es 0.0043) sería detectable ni con 50 repeticiones. Repetir el bucle externo no habría cambiado el veredicto.

**E4. Información frente a combinador.** Recuperar los códigos ICD-9 completos y el historial del paciente mejora 0.0055 en los 5 folds ($p = 0.010$), más que cualquier cambio del combinador probado (entre −0.0001 y +0.0024 frente a XGBoost). Esa cifra viene de otro modelo y de una partición propia, así que la comparación es orientativa, pero apunta al mismo lugar: el margen está en los datos.

**Cautela.** Aquí se probaron cinco variantes después del protocolo, sobre los mismos folds, sin corrección por comparaciones múltiples. Aunque alguna hubiera rozado la significancia, no podría presentarse como evidencia de superioridad. Ninguna lo hizo.

## 21.6. Todas las comprobaciones posteriores al protocolo

Esta tabla reúne las nueve verificaciones del capítulo 20 y las cuatro exploraciones de este capítulo. Las verificaciones preguntan si el resultado es correcto; las exploraciones, si DES-CPL puede mejorarse. Ninguna forma parte del protocolo pre-registrado y ninguna cambia el veredicto: DES-CPL no supera al mejor benchmark con significancia.

| # | Comprobación | Resultado clave | ¿Cambia el veredicto? |
|---|---|---|---|
| V1 | Reproducibilidad de la evaluación | Las métricas y configuraciones del capítulo 13 se reproducen exactamente (diferencia máxima 1e-16) | No |
| V2 | Orden de las filas en el SVM | Barajar cambia su AUC-PR en +0.0004 de media | No |
| V3 | Reproducción de Bayes | La diferencia de hasta 0.0008 con el capítulo 12 viene del redondeo al guardar sus predicciones; sin efecto en DES-CPL | No |
| V4 | Nueve modelos en DES-CPL | 0.2302, +0.0012 frente a XGBoost ($p = 0.36$); el apilamiento llega a 0.2320 | No |
| V5 | Cinco pliegues en la selección | 0.2289, −0.0001 frente a XGBoost ($p = 0.51$) | No |
| V6 | Predicción conforme | En los folds externos la cobertura de los readmitidos es la nominal (0.896); la explicación de la subcobertura en prueba se retiró | No |
| V7 | Parada temprana de XGBoost | Separar la validación con pacientes reales sin repetir la búsqueda deja un XGBoost peor (−0.054) y degenerado en dos folds; no es una versión comparable | No |
| V8 | Calibración del SVM agrupada | +0.0004 de AUC-PR; el Brier no cambia | No |
| V9 | Duplicados en los conjuntos adicionales | En CDC, la ventaja de DES-CPL sobre XGBoost pasa de +0.0060 a +0.0058 sin los registros con copia | No |
| E1 | Cinco pliegues y nueve modelos | 0.2314, +0.0024 frente a XGBoost ($p = 0.24$); el apilamiento llega a 0.2333 | No |
| E2 | Competencia por ordenamiento | 0.2269 con la configuración oficial y 0.2309 con la de E1; $\lambda = 1$ en 7 de 10 folds | No |
| E3 | Potencia | La diferencia mínima detectable no baja de 0.0048 aunque se repita el bucle externo | No |
| E4 | Información frente a combinador | Los códigos ICD-9 completos y el historial mejoran 0.0055, más que cualquier cambio del combinador | No |

**Lectura conjunta.** Las verificaciones muestran que el resultado no depende de un error ni de una decisión discutible: corregir cada una deja a DES-CPL donde estaba o lo acerca a XGBoost sin superarlo. Las exploraciones muestran que tampoco hay una versión mejor al alcance: las variantes más fuertes suben unas milésimas, pero los ensambles estáticos con los mismos modelos suben lo mismo o más, y ninguna diferencia de ese tamaño sería detectable con más repeticiones. El límite está en la información disponible, no en el combinador ni en el diseño de la evaluación.